# 02 - Multiple Linear Regression End-to-End

This notebook teaches multiple linear regression using numeric and categorical predictors.

The flow is:

1. Problem framing
2. EDA
3. Train-test split
4. One-hot encoding
5. scikit-learn pipeline
6. Metrics and visual evaluation
7. Coefficient interpretation
8. statsmodels inference
9. Cross-validation


## 1. Problem statement

We want to predict `sales_k_units` from marketing and business drivers:

- TV spend
- Radio spend
- Social spend
- Price index
- Competitor spend
- Season

Multiple regression equation:

```text
sales = beta_0 + beta_1*x_1 + beta_2*x_2 + ... + error
```

Coefficient interpretation must use:

> holding other variables constant


## 2. Import libraries


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

DATA_DIR = Path.cwd().parent / "data" if (Path.cwd().parent / "data").exists() else Path.cwd() / "data"


## 3. Load data


In [ ]:
df = pd.read_csv(DATA_DIR / "multiple_regression_marketing_sales.csv")
df.head()


## 4. Data audit


In [ ]:
pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "unique_values": df.nunique().values,
})


In [ ]:
df.describe(include='all').T


## 5. Visualize average sales by season


In [ ]:
season_sales = df.groupby("season")["sales_k_units"].mean().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(season_sales.index, season_sales.values)
plt.title("Average sales by season")
plt.xlabel("Season")
plt.ylabel("Average sales, thousand units")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 6. Visualize numeric feature relationships


In [ ]:
numeric_features = ["tv_spend_k", "radio_spend_k", "social_spend_k", "price_index", "competitor_spend_k"]
target = "sales_k_units"

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for idx, feature in enumerate(numeric_features):
    axes[idx].scatter(df[feature], df[target], alpha=0.7)
    axes[idx].set_title(f"{feature} vs sales")
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel(target)
    axes[idx].grid(True, alpha=0.3)

axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 7. Correlation heatmap


In [ ]:
corr = df[numeric_features + [target]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation heatmap")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 8. Define features and target


In [ ]:
categorical_features = ["season"]

X = df[numeric_features + categorical_features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 9. Train-test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=43,
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


## 10. Preprocessing and modelling pipeline


In [ ]:
preprocess = ColumnTransformer([
    ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ("numeric", "passthrough", numeric_features),
])

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression()),
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)


## 11. Model metrics


In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae, rmse, r2],
})


## 12. Actual vs predicted plot


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.8)
line_min = min(y_test.min(), pred.min())
line_max = max(y_test.max(), pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linewidth=2)
plt.title("Actual vs predicted sales")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.grid(True, alpha=0.3)
plt.show()


## 13. Coefficient table


In [ ]:
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
coefficients = pipeline.named_steps["model"].coef_

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
}).sort_values("coefficient", ascending=False)

coef_table


## 14. Coefficient visualization


In [ ]:
coef_plot = coef_table.sort_values("coefficient")

plt.figure(figsize=(9, 5))
plt.barh(coef_plot["feature"], coef_plot["coefficient"])
plt.axvline(0, linewidth=1)
plt.title("Multiple regression coefficients")
plt.xlabel("Coefficient value")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 15. Error analysis table


In [ ]:
error_table = X_test.copy()
error_table["actual_sales"] = y_test.values
error_table["predicted_sales"] = pred
error_table["residual"] = error_table["actual_sales"] - error_table["predicted_sales"]
error_table["absolute_error"] = error_table["residual"].abs()

error_table.sort_values("absolute_error", ascending=False).round(2).head(10)


## 16. Residual plot


In [ ]:
residuals = y_test - pred

plt.figure(figsize=(7, 5))
plt.scatter(pred, residuals, alpha=0.8)
plt.axhline(0, linewidth=1)
plt.title("Residuals vs predicted sales")
plt.xlabel("Predicted sales")
plt.ylabel("Residual")
plt.grid(True, alpha=0.3)
plt.show()


## 17. statsmodels OLS inference


In [ ]:
ols = smf.ols(
    "sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)",
    data=df,
).fit()

pd.DataFrame({
    "coefficient": ols.params,
    "p_value": ols.pvalues,
    "lower_95": ols.conf_int()[0],
    "upper_95": ols.conf_int()[1],
}).sort_values("p_value")


## 18. Cross-validation


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
scores = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"},
)

cv_results = pd.DataFrame({
    "fold": range(1, 6),
    "r2": scores["test_r2"],
    "mae": -scores["test_mae"],
    "rmse": -scores["test_rmse"],
})
cv_results.round(4)


## 19. Final interpretation

A sample conclusion:

> The model explains a high proportion of sales variation in this synthetic dataset. TV, radio, and social spend show positive associations with sales, while price index and competitor spend show negative associations, holding the other variables constant.


## Student exercises

1. Remove `season` and compare R².
2. Remove `competitor_spend_k` and compare coefficients.
3. Add a residual histogram.
4. Change the test size to 30%.
5. Write a business recommendation using the coefficient table.
